<a href="https://colab.research.google.com/github/Arczisork/Sztuczna-inteligencja/blob/main/RAG/RAG_Artur_11971.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q faiss-cpu pymupdf sentence-transformers tqdm ollama
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 63.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 88 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (22.2 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3

In [7]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 10
!ollama pull llama3.2:3b
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


In [8]:
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import ollama

embedder = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
model_id = "llama3.2:3b"

print("Embedding model załadowany")
print("Model Ollama:", model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model załadowany
Model Ollama: llama3.2:3b


In [9]:
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import ollama

embedder = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
model_id = "llama3.2:3b"

print("Embedding model załadowany")
print("Model Ollama:", model_id)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model załadowany
Model Ollama: llama3.2:3b


In [10]:
import os

os.makedirs("knowledge", exist_ok=True)
os.makedirs("vec_db", exist_ok=True)

print(os.getcwd())
print(os.listdir())

/content
['.config', 'ollama.log', 'knowledge', 'vec_db', 'sample_data']


In [12]:
os.makedirs("knowledge", exist_ok=True)
os.makedirs("vec_db", exist_ok=True)

pdf_files = [
    f for f in os.listdir("knowledge")
    if f.lower().endswith(".pdf")
]

print("Pliki PDF znalezione w katalogu knowledge:")

for pdf in pdf_files:
    print("-", pdf)

if len(pdf_files) == 0:
    raise Exception(
        "Brak plików PDF w katalogu knowledge. "
        "Dodaj PDF do repozytorium GitHub w folderze knowledge."
    )

Pliki PDF znalezione w katalogu knowledge:
- Specyfikacja_Istotnych_Warunkow_Zamowienia.pdf


In [13]:
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())
metadata = []

print("Liczba chunków:", index.ntotal)

Liczba chunków: 0


In [14]:
class Utils:
    def __init__(
        self,
        embedding_model=None,
        llm_model="llama3.2:3b",
        index=None,
        metadata=None,
        chunk_size=1000
    ):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        text = []
        pdf_document = pymupdf.open(pdf_path)

        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            page_text = str(page.get_text()).replace("\n", " ")
            text.append((page_num, page_text))

        return text

    def chunk_text(self, text):
        chunks = []

        for page_num, page_text in text:
            page_text = page_text.strip()

            for i in range(0, len(page_text), self.chunk_size):
                chunk = page_text[i:i+self.chunk_size].strip()

                if chunk:
                    chunks.append((page_num, chunk))

        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        os.makedirs(db_loc, exist_ok=True)

        for chunk_num, (page_number, chunk) in enumerate(
            tqdm(chunks, desc=f"Dodawanie {filename}")
        ):
            embedding = self.embedding_model.encode(
                chunk,
                show_progress_bar=False
            )

            embedding = np.array([embedding]).astype("float32")
            self.index.add(embedding)

            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })

        faiss.write_index(
            self.index,
            os.path.join(db_loc, "vector_database.index")
        )

        with open(
            os.path.join(db_loc, "metadata.json"),
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                self.metadata,
                file,
                ensure_ascii=False,
                indent=2
            )

    def process_file(self, file_path):
        if not file_path.lower().endswith(".pdf"):
            print("Pomijam plik:", file_path)
            return 0

        text = self.extract_text_from_pdf(file_path)
        chunks = self.chunk_text(text)

        self.add_chunks_to_faiss(
            chunks,
            filename=os.path.basename(file_path)
        )

        return len(chunks)

    def answer_question(
        self,
        prompt_template,
        query,
        max_tokens=512,
        temp=0.1,
        k=10
    ):
        question_embedding = self.embedding_model.encode(
            query,
            show_progress_bar=False
        )

        question_embedding = np.array([question_embedding]).astype("float32")

        D, I = self.index.search(question_embedding, k)

        chunks = [
            self.metadata[i]
            for i in I[0]
            if i != -1
        ]

        context = ""

        for i, chunk in enumerate(chunks):
            context += (
                f"[{i+1}] Plik: {chunk['filename']}, "
                f"strona {chunk['page_number'] + 1}\n"
            )
            context += chunk["chunk"] + "\n\n"

        prompt = prompt_template.format(
            context=context,
            query=query
        )

        response = ollama.chat(
            model=self.llm_model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Odpowiadaj tylko na podstawie podanego kontekstu. "
                        "Nie wymyślaj informacji. "
                        "Jeśli kontekst nie zawiera odpowiedzi, napisz: "
                        "Kontekst nie zawiera odpowiedzi na to pytanie."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": temp,
                "num_predict": max_tokens
            }
        )

        return response["message"]["content"], chunks

In [15]:
utils = Utils(
    embedding_model=embedder,
    llm_model=model_id,
    index=index,
    metadata=metadata,
    chunk_size=1000
)

print("RAG gotowy")

RAG gotowy


In [16]:
knowledge_dir = "knowledge"

pdf_files = [
    f for f in os.listdir(knowledge_dir)
    if f.lower().endswith(".pdf")
]

for file in pdf_files:
    print("Przetwarzam:", file)
    utils.process_file(os.path.join(knowledge_dir, file))

print("Liczba chunków:", index.ntotal)

Przetwarzam: Specyfikacja_Istotnych_Warunkow_Zamowienia.pdf


Dodawanie Specyfikacja_Istotnych_Warunkow_Zamowienia.pdf: 100%|██████████| 152/152 [00:03<00:00, 46.54it/s]

Liczba chunków: 152


In [17]:
prompt_template = """
Kontekst:
{context}

Pytanie:
{query}

Odpowiedz po polsku, krótko i konkretnie.
Korzystaj wyłącznie z podanego kontekstu.
Jeżeli odpowiedzi nie ma w kontekście, napisz:
Kontekst nie zawiera odpowiedzi na to pytanie.
"""

In [18]:
answer, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query="Zamówienie musi być wykonane w terminie do ilu miesięcy od dnia podpisania umowy?",
    k=10
)

print("ODPOWIEDŹ:")
print(answer)

ODPOWIEDŹ:
W terminie 7 dni od dnia podpisania umowy.


In [19]:
print("ŹRÓDŁA:")

for i, c in enumerate(chunks):
    print(f"\n--- ŹRÓDŁO {i+1} ---")
    print("Plik:", c["filename"])
    print("Strona:", c["page_number"] + 1)
    print(c["chunk"][:700])

ŹRÓDŁA:

--- ŹRÓDŁO 1 ---
Plik: Specyfikacja_Istotnych_Warunkow_Zamowienia.pdf
Strona: 18
dni – jeżeli zostały przesłane w inny sposób;  2)  w terminie 5 dni od dnia zamieszczenia ogłoszenia w Biuletynie Zamówień Publicznych -  wobec treści ogłoszenia o zamówieniu;  3)  w terminie 5 dni od dnia zamieszczenia Specyfikacji Istotnych Warunków Zamówienia   na stronie internetowej  - wobec treści Specyfikacji Istotnych Warunków Zamówienia;   4)  w terminie 5 dni od dnia, w którym powzięto lub przy zachowaniu należytej staranności  można było powziąć wiadomość o okolicznościach stanowiących podstawę jego  wniesienia- wobec czynności innych niż określone w pkt 1-3.  8.  Odwołujący przesyła kopie odwołania Zamawiającemu przed upływem terminu do wniesienia odwołania  w taki sposób, aby mó

--- ŹRÓDŁO 2 ---
Plik: Specyfikacja_Istotnych_Warunkow_Zamowienia.pdf
Strona: 26
2.  Wykonawca sporządza Protokoły Odbioru wg wzoru stanowiącego załącznik nr 2 do Umowy i dostarcza  go Zamawiającemu w 2 jedno

In [20]:
questions = [
    "Jaka była wysokość wadium?",
    "Jakie były kryteria oceny ofert?",
    "Jakie było znaczenie ceny w ocenie ofert?",
    "Jaki był termin wykonania zamówienia?",
    "Jakie instytucje miały zostać zintegrowane przez system?",
    "Jakie wymagania miał spełniać kierownik projektu?"
]

for q in questions:
    answer, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=q,
        k=10
    )

    print("=" * 80)
    print("PYTANIE:", q)
    print("ODPOWIEDŹ:", answer)
    print("ŹRÓDŁA:", [c["page_number"] + 1 for c in chunks[:3]])

PYTANIE: Jaka była wysokość wadium?
ODPOWIEDŹ: Wysokość wadium wynosi 6 000,00 zł.
ŹRÓDŁA: [18, 46, 33]
PYTANIE: Jakie były kryteria oceny ofert?
ODPOWIEDŹ: Cena oferty (80%) oraz oferowany okres gwarancji (20%).
ŹRÓDŁA: [16, 17, 19]
PYTANIE: Jakie było znaczenie ceny w ocenie ofert?
ODPOWIEDŹ: Cena oferty miała znaczenie 80%.
ŹRÓDŁA: [16, 15, 21]
PYTANIE: Jaki był termin wykonania zamówienia?
ODPOWIEDŹ: W terminie określonym w art. 94 ustawy.
ŹRÓDŁA: [18, 4, 26]
PYTANIE: Jakie instytucje miały zostać zintegrowane przez system?
ODPOWIEDŹ: Instytucje, które miały zostać zintegrowane przez system, to:

- Centralny Ośrodek Sportu (COS)
- Główny Urząd Statystyczny (GUS)
ŹRÓDŁA: [46, 40, 41]
PYTANIE: Jakie wymagania miał spełniać kierownik projektu?
ODPOWIEDŹ: Wykształcenie wyższe techniczne lub informatyczne.
ŹRÓDŁA: [51, 51, 33]
